### Constructors: Bronze to Silver
Clean and transform raw constructors data from `formula1_incr.bronze.constructors` into `formula1_incr.silver.constructors`.

#### Setup
- `01.environment-config` → loads catalog name, bronze/silver schema names
- `03.silver_helpers` → loads the `write_to_silver()` function we use to save data

In [0]:
%run ../00-common/01.environment-config 

In [0]:
%run ../00-common/03.silver_helpers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

In [0]:
dbutils.widgets.text('p_batch_id','')
v_batch_id= dbutils.widgets.get('p_batch_id')

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.constructors'
silver_table = f'{catalog_name}.{silver_schema}.constructors'

#### Read Bronze
- Read raw data from the bronze table filtered by `batch_id`

In [0]:
constructors_df = spark.read.table(bronze_table).filter(col('batch_id') == v_batch_id)
display(constructors_df)


#### Drop Columns
- Remove `url` column — not needed for analysis

In [0]:
constructors_drop_df = constructors_df.drop('url')

#### Rename Columns
- Convert camelCase to snake_case for consistency

In [0]:
constructor_renamed_df = (constructors_drop_df.withColumnsRenamed({'constructorId': 'constructor_id',
 'name': 'constructor_name'}))                                         

#### Remove Duplicates
- Keep one row per constructor using `dropDuplicates()`

In [0]:
constructor_distinct_df = constructor_renamed_df.dropDuplicates(["constructor_id"])


In [0]:
display(constructor_distinct_df)

#### Title Case
- Apply `initcap()` to `nationality` so it looks clean

In [0]:
from pyspark.sql.functions import *
constructor_final_df = (constructor_distinct_df
                        .withColumn('nationality',initcap(col('nationality'))))

In [0]:
write_to_silver(
    input_df = constructor_final_df,
    target_table = silver_table,
    merge_condition = 's.constructor_id = t.constructor_id',
    columns_to_update =[
        'constructor_id',
        'constructor_name',
        'nationality',
        'ingestion_timestamp',
        'source_file',
        'batch_id']
)

#### Write to Silver
- If the silver table doesn't exist yet, it creates it from scratch
- If it already exists, it merges new/updated rows using `write_to_silver()` (insert new, update changed)

In [0]:
spark.read.table(silver_table).display()